In [ ]:

# ── Colab setup ──────────────────────────────────────────────────────────────
# When running locally this cell does nothing.
# When running in Google Colab, clones the repo so all data files are available.
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists('raw_matches.csv'):
    REPO_URL = 'https://github.com/emaforlin/world_cup_learning_modernized.git'
    !git clone {REPO_URL} /content/wcl
    os.chdir('/content/wcl')
    sys.path.insert(0, '/content/wcl')


In [ ]:
import utils

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Normalization

In [ ]:
input_cols = ['year',
              'matches_won_percent',
              'podium_score_yearly',
              'matches_won_percent_2',
              'podium_score_yearly_2',]
output_col = 'winner'

In [ ]:
matches = utils.get_matches(with_team_stats=True, duplicate_with_reversed=True)
matches

In [ ]:
print("Matches per year (doubled dataset):")
print(matches.groupby('year').size().to_string())
print(f"\nTotal: {len(matches)} rows across {matches['year'].nunique()} tournaments")


In [ ]:
train, test = train_test_split(matches, test_size=0.2)

In [ ]:
network = Sequential([
    Input((len(input_cols),)),
    Normalization(),
    Dense(10, activation='sigmoid'),
    Dense(10, activation='sigmoid'),
    Dense(1, activation='sigmoid'),
])

network.layers[0].adapt(train[input_cols].values)

network.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy',],
)


network.summary()

In [ ]:
network.fit(
    train[input_cols], train[output_col],
    epochs=50,
    batch_size=128,
)

In [ ]:
train_predictions = network.predict(train[input_cols])

accuracy_score(train[output_col], train_predictions.round())

In [ ]:
test_predictions = network.predict(test[input_cols])

accuracy_score(test[output_col], test_predictions.round())

In [ ]:
def predict(year, team1, team2):
    case_inputs = utils.build_inputs_for_match(year, team1, team2, input_cols)
    result = network.predict(case_inputs)
    
    if result > 0.5:
        winner = team1
    else:
        winner = team2
        
    print(result[0][0], '→', winner)

In [ ]:
predict(1950, 'Mexico', 'Brazil')  # real result: 4-0 wins Brazil

In [ ]:
predict(1990, 'United Arab Emirates', 'Colombia')  # real result: 2-0 wins Colombia

In [ ]:
predict(2002, 'South Africa', 'Spain')  # real result: 2-3 wins Spain

In [ ]:
predict(2010, 'Japan', 'Cameroon')  # real result: 1-0 wins Japan

In [ ]:
predict(2014, 'Argentina', 'Germany')